# Anexo técnico: verificación del modelo v4 congelado

Este cuaderno acompaña al cuaderno evaluable `notebook_proyecto.ipynb`. Su modo predeterminado `verify` no reentrena el modelo. Usa el corpus ELG, el modelo y los manifiestos de la carpeta `literalevel_bigdata/entrega_profesor_v4` en Drive; quien lo ejecute en Colab debe tener acceso a esa carpeta y autorizar su propio montaje de Drive. El 29 de septiembre de 2026 se ejecutaron localmente las 11 celdas de cálculo con CPU, el corpus con SHA-256 comprobado y el v4 congelado, omitiendo solo la celda `%pip` de instalación propia de Colab. La validación interna reprodujo 144 documentos, Spearman 0,908398 y MAE 0,498511. Las pruebas ciegas con textos restringidos permanecen opcionales.


# LiteraLevel V4 — cuaderno reproducible

**Objetivo.** Verificar o reentrenar el modelo V4 de dificultad de prosa
inglesa mediante una ejecución transparente de principio a fin.

- Unidad de análisis: documento de prosa continua.
- Ventanas: 250 palabras; máximo 8 por documento durante entrenamiento y validación.
- Agregación: promedio simple de las predicciones de las ventanas.
- Escala interna: 0=A1, 1=A2, 2=B1, 3=B2, 4=C1, 5=C2.
- Escala pública: 0–2; Básico (<0.8), Intermedio (<1.6), Avanzado.

> La relación con CEFR es un objetivo de investigación. El sistema no es una
> certificación ni una equivalencia oficial CEFR. La prueba dirigida A1–A2
> falló sus criterios predefinidos y se informa de forma explícita.

## 1. Entorno

Se usan rangos compatibles con el entrenamiento original. La celda guarda
las versiones efectivamente usadas para que la ejecución pueda auditarse.
Reinicia el entorno solo si Colab lo solicita después de instalar.

In [ ]:
%pip install -q "transformers>=5.16,<6" "scikit-learn>=1.3" "scipy>=1.10" "pandas>=2.1" "safetensors>=0.4"

In [ ]:
import hashlib
import importlib.metadata as metadata
import json
import math
import os
import platform
import random
import re
import sys
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    get_linear_schedule_with_warmup,
)

SEED = 20260905

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True, warn_only=True)
    if hasattr(torch.backends, "cudnn"):
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True

seed_everything()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Python:", sys.version.split()[0])
print("Plataforma:", platform.platform())
print("Dispositivo:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
for package in ["torch", "transformers", "numpy", "pandas", "scipy", "scikit-learn"]:
    print(f"{package}: {metadata.version(package)}")

## 2. Configuración y verificación de entradas

`verify` carga el modelo congelado y reproduce la validación interna. `train`
inicia desde `distilroberta-base`, exige GPU y escribe en una carpeta nueva.
No hay claves ni contraseñas en el cuaderno.

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except ImportError:
    pass

RUN_MODE = os.environ.get("LITERALEVEL_RUN_MODE", "verify").lower()
if RUN_MODE not in {"verify", "train"}:
    raise ValueError("LITERALEVEL_RUN_MODE debe ser 'verify' o 'train'.")

DEFAULT_ROOT = "/content/drive/MyDrive/literalevel_bigdata/entrega_profesor_v4"
DELIVERY_ROOT = Path(os.environ.get("LITERALEVEL_DELIVERY_ROOT", DEFAULT_ROOT)).expanduser()
NOTEBOOK_DIR = DELIVERY_ROOT / "notebook"
DATA_DIR = DELIVERY_ROOT / "data"
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
OUTPUT_DIR = DELIVERY_ROOT / "outputs" / f"reproduccion_{RUN_ID}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)

DATA_PATH = DATA_DIR / "elg-cefr-en.json"
FROZEN_MODEL_DIR = DATA_DIR / "modelo_literalevel_v4_transformer"
EXTERNAL_BLIND_PATH = DATA_DIR / "validacion_externa" / "corpus_ciego_textos.jsonl"
EXTERNAL_KEY_PATH = DATA_DIR / "validacion_externa" / "clave_ciega.csv"
A1A2_BLIND_PATH = DATA_DIR / "validacion_a1a2" / "corpus_ciego_a1a2.jsonl"
A1A2_KEY_PATH = DATA_DIR / "validacion_a1a2" / "clave_ciega_a1a2.csv"

MANIFEST_PATH = NOTEBOOK_DIR / "inputs_manifest.json"
EXPECTED_RESULTS_PATH = NOTEBOOK_DIR / "expected_results.json"
print("Modo:", RUN_MODE)
print("Raíz:", DELIVERY_ROOT)
print("Salida nueva:", OUTPUT_DIR)

In [ ]:
def sha256_file(path: Path, chunk_size=1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()

manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
audit_rows = []
for item in manifest["inputs"]:
    path = DELIVERY_ROOT / item["path"]
    exists = path.is_dir() if item["kind"] == "directory" else path.is_file()
    status = "present" if exists else "absent_optional"
    details = ""
    if exists and item["kind"] == "file" and item.get("sha256"):
        actual = sha256_file(path)
        if actual.lower() != item["sha256"].lower():
            raise RuntimeError(f"SHA-256 inesperado para {path}: {actual}")
        details = actual
    if exists and item["kind"] == "directory":
        missing = [name for name in item.get("required_files", []) if not (path / name).is_file()]
        if missing:
            raise FileNotFoundError(f"Faltan archivos del modelo en {path}: {missing}")
        details = ", ".join(sorted(p.name for p in path.iterdir() if p.is_file()))
    audit_rows.append({"entrada": item["name"], "estado": status, "detalle": details})

if not DATA_PATH.is_file():
    raise FileNotFoundError(f"Falta el corpus redistribuible: {DATA_PATH}")
if RUN_MODE == "verify" and not FROZEN_MODEL_DIR.is_dir():
    raise FileNotFoundError(f"Falta el modelo congelado: {FROZEN_MODEL_DIR}")

audit = pd.DataFrame(audit_rows)
display(audit)
(OUTPUT_DIR / "auditoria_entradas.json").write_text(
    json.dumps(audit_rows, ensure_ascii=False, indent=2), encoding="utf-8"
)

## 3. Preparación de prosa y partición documental

Las ventanas de un mismo documento nunca se reparten entre entrenamiento y
validación. La longitud total y el número de ventanas no son entradas del
transformador. Se reproducen las reglas originales de V4.

In [ ]:
MODEL_NAME = "distilroberta-base"
BLOCK_WORDS = 250
MIN_WORDS = 150
MAX_BLOCKS_PER_DOC = 8
EPOCHS = 4
BATCH_SIZE = 16
LEARNING_RATE = 2e-5

LEVEL_VALUE = {
    "A1": 0.0, "A1+": 0.5, "A2": 1.0, "A2+": 1.5,
    "B1": 2.0, "B1+": 2.5, "B2": 3.0, "B2+": 3.5,
    "C1": 4.0, "C2": 5.0,
}
ANCHORS = ["A1", "A2", "B1", "B2", "C1", "C2"]
WORD_RE = re.compile(r"[A-Za-z]+(?:['’\-][A-Za-z]+)*")
SENT_RE = re.compile(r"[^.!?]+[.!?]+|[^.!?]+$")
BULLET_RE = re.compile(r"^\s*(?:[-*•▪◦]|\d+[.)]|[A-Z][.)])\s+")

def normalize(text):
    return re.sub(r"\s+", " ", str(text)).strip()

def words(text):
    return WORD_RE.findall(str(text))

def continuous_prose(text):
    raw = str(text).replace("\r", "\n")
    tokens = words(raw)
    if len(tokens) < MIN_WORDS:
        return False, "short"
    sentences = [s for s in SENT_RE.findall(normalize(raw)) if len(words(s)) >= 3]
    if len(sentences) < 4:
        return False, "few_sentences"
    mean_sentence = np.mean([len(words(s)) for s in sentences])
    if not 5 <= mean_sentence <= 55:
        return False, "sentence_shape"
    lines = [line.strip() for line in raw.splitlines() if line.strip()]
    bullet_lines = sum(bool(BULLET_RE.match(line)) for line in lines)
    short_lines = sum(len(words(line)) <= 4 for line in lines)
    if bullet_lines >= 2 or (len(lines) >= 6 and short_lines / len(lines) > 0.35):
        return False, "list_layout"
    return True, "ok"

def fixed_blocks(text, size=BLOCK_WORDS, minimum=MIN_WORDS, cap=MAX_BLOCKS_PER_DOC):
    tokens = words(text)
    n = len(tokens)
    if n < minimum:
        return []
    if n <= size:
        return [" ".join(tokens)]
    starts = list(range(0, n - size + 1, size))
    last_start = n - size
    if starts[-1] != last_start:
        starts.append(last_start)
    if len(starts) > cap:
        indexes = np.linspace(0, len(starts) - 1, cap).round().astype(int)
        starts = [starts[i] for i in sorted(set(indexes.tolist()))]
    return [" ".join(tokens[start:start + size]) for start in starts]

def public_score(value):
    return float(np.clip(value, 0, 5) * 0.4)

def public_band(score):
    return "basico" if score < 0.8 else "intermedio" if score < 1.6 else "avanzado"

In [ ]:
def load_documents(path=DATA_PATH):
    raw = json.loads(path.read_text(encoding="utf-8"))
    documents, rejected = [], Counter()
    for index, item in enumerate(raw):
        level = str(item.get("cefr_level", "")).upper().strip()
        if level not in LEVEL_VALUE:
            rejected["unknown_level"] += 1
            continue
        text = normalize(item.get("text", ""))
        ok, reason = continuous_prose(text)
        if not ok:
            rejected[reason] += 1
            continue
        blocks = fixed_blocks(text)
        if not blocks:
            rejected["no_blocks"] += 1
            continue
        documents.append({
            "doc_id": str(item.get("title") or f"doc_{index}"),
            "level": level,
            "y": LEVEL_VALUE[level],
            "text": text,
            "blocks": blocks,
        })
    return documents, rejected

def split_documents(documents):
    strata = [ANCHORS[int(np.clip(round(doc["y"]), 0, 5))] for doc in documents]
    return train_test_split(
        documents, test_size=0.22, random_state=SEED, stratify=strata
    )

def flatten(documents, training=False):
    counts = Counter(ANCHORS[int(np.clip(round(doc["y"]), 0, 5))] for doc in documents)
    rows = []
    for doc in documents:
        anchor = ANCHORS[int(np.clip(round(doc["y"]), 0, 5))]
        weight = (1.0 / len(doc["blocks"])) * math.sqrt(len(documents) / (6 * counts[anchor]))
        for block in doc["blocks"]:
            rows.append({
                "text": block, "y": doc["y"], "doc_id": doc["doc_id"],
                "level": doc["level"], "weight": weight if training else 1.0,
            })
    return rows

documents, rejected = load_documents()
train_docs, validation_docs = split_documents(documents)
train_rows = flatten(train_docs, training=True)
validation_rows = flatten(validation_docs, training=False)

dataset_summary = {
    "raw_documents": 712,
    "accepted_documents": len(documents),
    "rejected": dict(rejected),
    "train_documents": len(train_docs),
    "validation_documents": len(validation_docs),
    "train_blocks": len(train_rows),
    "validation_blocks": len(validation_rows),
    "levels": dict(Counter(doc["level"] for doc in documents)),
}
print(json.dumps(dataset_summary, ensure_ascii=False, indent=2))
assert len(documents) == 654 and len(train_docs) == 510 and len(validation_docs) == 144
assert set(doc["doc_id"] for doc in train_docs).isdisjoint(doc["doc_id"] for doc in validation_docs)

## 4. Modelo, entrenamiento opcional e inferencia

En `verify` se usa `local_files_only=True`: no se descarga ni sustituye el
modelo. En `train`, el resultado se guarda como `modelo_reentrenado_v4`
dentro de la salida nueva.

In [ ]:
class BlockDataset(Dataset):
    def __init__(self, rows, tokenizer):
        self.rows = rows
        self.tokenizer = tokenizer
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, index):
        row = self.rows[index]
        encoded = self.tokenizer(row["text"], truncation=True, max_length=256)
        encoded["labels"] = float(row["y"])
        encoded["weights"] = float(row["weight"])
        encoded["row_index"] = index
        return encoded

class Collator:
    def __init__(self, tokenizer):
        self.base = DataCollatorWithPadding(tokenizer, return_tensors="pt")
    def __call__(self, features):
        weights = torch.tensor([item.pop("weights") for item in features], dtype=torch.float32)
        row_index = torch.tensor([item.pop("row_index") for item in features], dtype=torch.long)
        batch = self.base(features)
        batch["weights"] = weights
        batch["row_index"] = row_index
        return batch

def make_loader(rows, tokenizer, shuffle=False, batch_size=BATCH_SIZE):
    generator = torch.Generator().manual_seed(SEED)
    return DataLoader(
        BlockDataset(rows, tokenizer), batch_size=batch_size, shuffle=shuffle,
        num_workers=0, pin_memory=DEVICE.type == "cuda", generator=generator,
        collate_fn=Collator(tokenizer),
    )

@torch.inference_mode()
def predict_rows(model, loader, rows):
    model.eval()
    predictions = np.zeros(len(rows), dtype=float)
    for batch in loader:
        indexes = batch.pop("row_index").numpy()
        batch.pop("weights", None)
        batch.pop("labels", None)
        inputs = {key: value.to(DEVICE) for key, value in batch.items()}
        output = model(**inputs).logits.squeeze(-1).float().cpu().numpy()
        predictions[indexes] = np.clip(output, 0, 5)
    return predictions

def document_metrics(rows, block_predictions):
    frame = pd.DataFrame(rows)[["doc_id", "level", "y"]].copy()
    frame["block_prediction"] = block_predictions
    docs = frame.groupby("doc_id", as_index=False).agg(
        level=("level", "first"), y=("y", "first"),
        predicted=("block_prediction", "mean"), n_blocks=("block_prediction", "size"),
    )
    true_anchor = np.clip(np.rint(docs["y"]), 0, 5).astype(int)
    predicted_anchor = np.clip(np.rint(docs["predicted"]), 0, 5).astype(int)
    result = {
        "n_documents": int(len(docs)),
        "exact_anchor_accuracy": float(np.mean(true_anchor == predicted_anchor)),
        "within_one_anchor": float(np.mean(np.abs(true_anchor - predicted_anchor) <= 1)),
        "mae_cefr_steps": float(np.mean(np.abs(docs["y"] - docs["predicted"]))),
        "spearman": float(spearmanr(docs["y"], docs["predicted"]).statistic),
    }
    docs["score_0_2"] = docs["predicted"].map(public_score)
    docs["band"] = docs["score_0_2"].map(public_band)
    return result, docs

In [ ]:
def train_model(train_rows, validation_rows):
    if DEVICE.type != "cuda":
        raise RuntimeError("El reentrenamiento requiere una GPU; selecciona T4 en Colab.")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=1, problem_type="regression"
    ).to(DEVICE)
    train_loader = make_loader(train_rows, tokenizer, shuffle=True)
    validation_loader = make_loader(validation_rows, tokenizer)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
    total_steps = EPOCHS * len(train_loader)
    scheduler = get_linear_schedule_with_warmup(
        optimizer, max(1, total_steps // 10), total_steps
    )
    scaler = torch.amp.GradScaler("cuda")
    best_criterion, best_state, history = None, None, []
    for epoch in range(1, EPOCHS + 1):
        model.train()
        losses = []
        for batch in train_loader:
            batch.pop("row_index")
            weights = batch.pop("weights").to(DEVICE)
            labels = batch.pop("labels").to(DEVICE).float()
            inputs = {key: value.to(DEVICE) for key, value in batch.items()}
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                prediction = model(**inputs).logits.squeeze(-1)
                per_item = torch.nn.functional.smooth_l1_loss(prediction, labels, reduction="none")
                loss = (per_item * weights).sum() / weights.sum()
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            losses.append(float(loss.detach().cpu()))
        predictions = predict_rows(model, validation_loader, validation_rows)
        metrics_epoch, _ = document_metrics(validation_rows, predictions)
        record = {"epoch": epoch, "train_loss": float(np.mean(losses)), **metrics_epoch}
        history.append(record)
        print(json.dumps(record, ensure_ascii=False))
        criterion = metrics_epoch["mae_cefr_steps"] - 0.15 * metrics_epoch["spearman"]
        if best_criterion is None or criterion < best_criterion:
            best_criterion = criterion
            best_state = {name: value.detach().cpu().clone() for name, value in model.state_dict().items()}
    model.load_state_dict(best_state)
    return model, tokenizer, history

if RUN_MODE == "train":
    model, tokenizer, training_history = train_model(train_rows, validation_rows)
    model_dir_used = OUTPUT_DIR / "modelo_reentrenado_v4"
    model_dir_used.mkdir(parents=True)
    model.save_pretrained(model_dir_used)
    tokenizer.save_pretrained(model_dir_used)
else:
    model_dir_used = FROZEN_MODEL_DIR
    tokenizer = AutoTokenizer.from_pretrained(model_dir_used, local_files_only=True)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_dir_used, local_files_only=True
    ).to(DEVICE)
    training_history = None
model.eval()
print("Modelo usado:", model_dir_used)

## 5. Validación interna reproducida

Esta evaluación usa exactamente la partición documental determinada por la
semilla 20260905. Se comparan las métricas recalculadas con las publicadas,
sin exigir igualdad bit a bit entre hardware distinto.

In [ ]:
validation_loader = make_loader(validation_rows, tokenizer, batch_size=32 if DEVICE.type == "cuda" else 8)
validation_predictions = predict_rows(model, validation_loader, validation_rows)
internal_metrics, internal_documents = document_metrics(validation_rows, validation_predictions)
internal_documents.to_csv(OUTPUT_DIR / "validacion_interna_reproducida.csv", index=False)
print(json.dumps(internal_metrics, ensure_ascii=False, indent=2))

expected = json.loads(EXPECTED_RESULTS_PATH.read_text(encoding="utf-8"))
reference = expected["internal_validation"]
comparison = pd.DataFrame([
    {"métrica": "spearman", "reproducida": internal_metrics["spearman"], "publicada": reference["spearman"]},
    {"métrica": "mae_cefr_steps", "reproducida": internal_metrics["mae_cefr_steps"], "publicada": reference["mae_cefr_steps"]},
    {"métrica": "exact_anchor_accuracy", "reproducida": internal_metrics["exact_anchor_accuracy"], "publicada": reference["exact_anchor_accuracy"]},
    {"métrica": "within_one_anchor", "reproducida": internal_metrics["within_one_anchor"], "publicada": reference["within_one_anchor"]},
])
comparison["diferencia_absoluta"] = (comparison["reproducida"] - comparison["publicada"]).abs()
display(comparison)
comparison.to_csv(OUTPUT_DIR / "comparacion_metricas_publicadas.csv", index=False)

## 6. Validaciones ciegas opcionales

La inferencia se realiza sobre el JSONL anónimo antes de abrir la clave. Las
predicciones se guardan junto con su SHA-256. Los textos restringidos no se
incluyen en la entrega; si no están presentes, la sección queda marcada
como omitida y se muestran las cifras publicadas en `expected_results.json`.

In [ ]:
@torch.inference_mode()
def score_blind_jsonl(path: Path, output_csv: Path):
    records = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]
    flat_rows = []
    for record in records:
        blocks = fixed_blocks(record["text"])
        if not blocks:
            raise ValueError(f"{record['id']} no cumple el mínimo de palabras")
        for block in blocks:
            flat_rows.append({
                "doc_id": record["id"], "level": "blind", "y": 0.0,
                "text": block, "weight": 1.0,
            })
    loader = make_loader(flat_rows, tokenizer, batch_size=32 if DEVICE.type == "cuda" else 8)
    predictions = predict_rows(model, loader, flat_rows)
    frame = pd.DataFrame({
        "id": [row["doc_id"] for row in flat_rows],
        "block_prediction": predictions,
    }).groupby("id", as_index=False).agg(
        predicted=("block_prediction", "mean"), n_blocks=("block_prediction", "size")
    )
    frame["score_0_2"] = frame["predicted"].map(public_score)
    frame["band"] = frame["score_0_2"].map(public_band)
    frame.to_csv(output_csv, index=False)
    return frame, sha256_file(output_csv)

def evaluate_keyed(predictions, key, id_column, level_column="level", target_column="target"):
    merged = key.merge(predictions, left_on=id_column, right_on="id", validate="one_to_one")
    error = np.abs(merged[target_column] - merged["predicted"])
    rounded = np.clip(np.rint(merged["predicted"]), 0, 5).astype(int)
    per_level_mae = merged.assign(abs_error=error).groupby(level_column)["abs_error"].mean()
    return {
        "documents": int(len(merged)),
        "spearman": float(spearmanr(merged[target_column], merged["predicted"]).statistic),
        "mae": float(error.mean()),
        "macro_mae": float(per_level_mae.mean()),
        "exact_anchor_accuracy": float(np.mean(rounded == merged[target_column])),
        "within_one_anchor": float(np.mean(np.abs(rounded - merged[target_column]) <= 1)),
    }, merged

In [ ]:
optional_results = {}

if EXTERNAL_BLIND_PATH.is_file() and EXTERNAL_KEY_PATH.is_file():
    external_predictions, external_pred_sha = score_blind_jsonl(
        EXTERNAL_BLIND_PATH, OUTPUT_DIR / "predicciones_validacion_externa.csv"
    )
    external_key = pd.read_csv(EXTERNAL_KEY_PATH)
    external_metrics, external_joined = evaluate_keyed(
        external_predictions, external_key, id_column="blind_id"
    )
    external_joined.to_csv(OUTPUT_DIR / "validacion_externa_desvelada.csv", index=False)
    optional_results["external_blind_validation"] = {
        **external_metrics, "predictions_sha256": external_pred_sha
    }
    print("Validación externa recalculada:")
    print(json.dumps(optional_results["external_blind_validation"], indent=2))
else:
    optional_results["external_blind_validation"] = {
        "status": "skipped_restricted_inputs_absent",
        "published_result": expected["external_blind_validation"],
    }
    print("Validación externa omitida: los textos restringidos no forman parte de la entrega.")

if A1A2_BLIND_PATH.is_file() and A1A2_KEY_PATH.is_file():
    a1a2_predictions, a1a2_pred_sha = score_blind_jsonl(
        A1A2_BLIND_PATH, OUTPUT_DIR / "predicciones_validacion_a1a2.csv"
    )
    a1a2_key = pd.read_csv(A1A2_KEY_PATH)
    a1a2_metrics, a1a2_joined = evaluate_keyed(
        a1a2_predictions, a1a2_key, id_column="id"
    )
    a1a2_metrics["auc"] = float(roc_auc_score(
        (a1a2_joined["level"] == "A2").astype(int), a1a2_joined["predicted"]
    ))
    means = a1a2_joined.groupby("level")["predicted"].mean()
    a1a2_metrics["mean_gap_a2_minus_a1"] = float(means["A2"] - means["A1"])
    a1a2_metrics["verdict"] = "PASS" if (
        a1a2_metrics["auc"] >= 0.80
        and a1a2_metrics["mean_gap_a2_minus_a1"] >= 0.35
        and a1a2_metrics["macro_mae"] <= 0.75
        and a1a2_metrics["within_one_anchor"] >= 0.90
    ) else "FAIL"
    a1a2_joined.to_csv(OUTPUT_DIR / "validacion_a1a2_desvelada.csv", index=False)
    optional_results["targeted_a1a2_validation"] = {
        **a1a2_metrics, "predictions_sha256": a1a2_pred_sha
    }
    print("Validación A1–A2 recalculada:")
    print(json.dumps(optional_results["targeted_a1a2_validation"], indent=2))
else:
    optional_results["targeted_a1a2_validation"] = {
        "status": "skipped_restricted_inputs_absent",
        "published_result": expected["targeted_a1a2_validation"],
    }
    print("Validación A1–A2 omitida: los textos restringidos no forman parte de la entrega.")

## 7. Informe de ejecución

Se guardan configuración, versiones, métricas y hashes en JSON. Este archivo
permite revisar qué se ejecutó sin depender de la salida visual del cuaderno.

In [ ]:
environment = {
    "python": sys.version,
    "platform": platform.platform(),
    "device": str(DEVICE),
    "gpu": torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else None,
    "packages": {
        name: metadata.version(name)
        for name in ["torch", "transformers", "numpy", "pandas", "scipy", "scikit-learn"]
    },
}
model_hashes = {
    path.name: sha256_file(path)
    for path in sorted(model_dir_used.iterdir())
    if path.is_file() and path.suffix in {".json", ".safetensors", ".bin"}
}
execution_report = {
    "status": "completed",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "run_mode": RUN_MODE,
    "seed": SEED,
    "design": {
        "base_model": MODEL_NAME,
        "block_words": BLOCK_WORDS,
        "minimum_words": MIN_WORDS,
        "maximum_blocks_per_document": MAX_BLOCKS_PER_DOC,
        "document_aggregation": "unweighted mean",
        "split_unit": "document",
        "uses_total_document_length": False,
    },
    "dataset": dataset_summary,
    "internal_validation": internal_metrics,
    "optional_validations": optional_results,
    "model_file_hashes": model_hashes,
    "environment": environment,
    "training_history": training_history,
}
report_path = OUTPUT_DIR / "reporte_reproduccion_v4.json"
report_path.write_text(json.dumps(execution_report, ensure_ascii=False, indent=2), encoding="utf-8")
print("Ejecución terminada.")
print("Reporte:", report_path)
print("SHA-256 del reporte:", sha256_file(report_path))

## Interpretación final

La validación externa principal de 76 textos fue fuerte, pero la prueba
independiente y dirigida de 28 fragmentos A1–A2 mostró una limitación real:
V4 no separa esos dos niveles editoriales con la fiabilidad exigida. Por
ello el uso defendible es **ordenar dificultad literaria en la escala propia
0–2**, mostrando incertidumbre entre fragmentos, y no prometer niveles CEFR.